In [1]:
import pandas as pd
import os

In [2]:
import torch

print("CUDA version in PyTorch:", torch.version.cuda)
print("Is CUDA available?", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA version in PyTorch: 12.1
Is CUDA available? True
NVIDIA GeForce RTX 3090


In [3]:
matches = pd.read_csv("../feature_creation/data/atp_matches_model_ready.csv")

In [4]:
list(matches.columns)


['tournament_id',
 'tournament_location',
 'draw_size',
 'tournament_date',
 'match_num',
 'score',
 'best_of',
 'round',
 'minutes',
 'w_ace',
 'w_df',
 'w_svpt',
 'w_1stIn',
 'w_1stWon',
 'w_2ndWon',
 'w_SvGms',
 'w_bpSaved',
 'w_bpFaced',
 'l_ace',
 'l_df',
 'l_svpt',
 'l_1stIn',
 'l_1stWon',
 'l_2ndWon',
 'l_SvGms',
 'l_bpSaved',
 'l_bpFaced',
 'year',
 'shortened_winner_name',
 'shortened_loser_name',
 'match_id',
 'Date',
 'W1',
 'W2',
 'W3',
 'W4',
 'W5',
 'L1',
 'L2',
 'L3',
 'L4',
 'L5',
 'Wsets',
 'Lsets',
 'Comment',
 'elo_pwin',
 'surface_elo_pwin',
 'blended_elo_pwin',
 'elo_diff',
 'surface_elo_diff',
 'blended_elo_diff',
 'surface_Clay',
 'surface_Grass',
 'surface_Hard',
 'tournament_level_ATP250',
 'tournament_level_ATP500',
 'tournament_level_Grand Slam',
 'tournament_level_Masters 1000',
 'outdoor',
 'target',
 'player1_id',
 'player2_id',
 'player1_name',
 'player2_name',
 'player1_ht',
 'player2_ht',
 'player1_ioc',
 'player2_ioc',
 'player1_age',
 'player2_age',
 

In [5]:
matches.shape


(22092, 128)

In [6]:
matches['Date'] = pd.to_datetime(matches['Date'])
matches['year'] = matches['Date'].dt.year
matches = matches[matches['year'] > 2014]

In [7]:
train_matches = matches[matches['year'] < 2022]
val_matches = matches[matches['year'] == 2022]
test_matches = matches[matches['year'] == 2023]

In [8]:
player1_info_cols = ['player1_surface_elo', 'player1_elo', 'player1_is_right_handed', 'player1_entry_LL', 'player1_entry_Q', 'player1_entry_WC',
                     'player1_ht', 'player1_age','player1_fatigue_score', 'player1_h2h_wins',
                     'player1_best_result_tournament_history', 'player1_last_result_tournament_history', 'player1_average_result_tournament_history',
                     'player1_round_level_win_pct', 'player1_round_level_appearances']
player2_info_cols = ['player2_surface_elo', 'player2_elo', 'player2_is_right_handed', 'player2_entry_LL', 'player2_entry_Q', 'player2_entry_WC',
                     'player2_ht', 'player2_age','player2_fatigue_score', 'player2_h2h_wins',
                     'player2_best_result_tournament_history', 'player2_last_result_tournament_history', 'player2_average_result_tournament_history',
                     'player2_round_level_win_pct', 'player2_round_level_appearances']

env_cols = ['tournament_level_ATP250', 'tournament_level_ATP500', 'tournament_level_Masters 1000',
         'tournament_level_Grand Slam','outdoor', 'surface_Grass', 'surface_Hard', 'surface_Clay']

## GRU for form

In [9]:
import torch.nn as nn

class GRUFormEncoder(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        gru_out, _ = self.gru(x)

        context = gru_out[:, -1, :]
        return context


In [10]:
class PairwiseFormEncoder(nn.Module):
    def __init__(self, form_dim: int, hidden_size: int, dropout: float = 0.0):
        super().__init__()
        self.encoder = GRUFormEncoder(input_size=4 * form_dim,
                                        hidden_size=hidden_size,
                                        dropout=dropout)

    def forward(self, p1_seq: torch.Tensor, p2_seq: torch.Tensor):
        diff = p1_seq - p2_seq
        prod = p1_seq * p2_seq
        z = torch.cat([p1_seq, p2_seq, diff, prod], dim=-1)
        return self.encoder(z)


In [11]:
import torch.nn.functional as F

class SymmetricNnWideDeep(nn.Module):
    def __init__(self,
                 player_info_feature_size: int,
                 env_feature_size: int,
                 dropout: float = 0.35,
                 bottleneck: int = 512,
                 hidden_mid: int = 256,
                 hidden_small: int = 128,
                 ):
        super().__init__()


        self.deep_in  = player_info_feature_size * 3 + env_feature_size

        self.wide = nn.Linear(self.deep_in, 1, bias=False)

        # deep layers split so we can return latent
        self.deep_fc1 = nn.Linear(self.deep_in, bottleneck)
        self.deep_fc2 = nn.Linear(bottleneck, hidden_mid)
        self.deep_fc3 = nn.Linear(hidden_mid, hidden_small)
        self.deep_out = nn.Linear(hidden_small, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, p1_info_f, p2_info_f, env_f, return_latent: bool = False):

        diff_f = p1_info_f - p2_info_f

        x1 = torch.cat([p1_info_f, p2_info_f,  diff_f, env_f], dim=1)
        x2 = torch.cat([p2_info_f, p1_info_f,  -diff_f, env_f], dim=1)

        # deep path (return latent before deep_out)
        h1 = self.dropout(F.relu(self.deep_fc1(x1)))
        h1 = self.dropout(F.relu(self.deep_fc2(h1)))
        h1_small = self.dropout(F.relu(self.deep_fc3(h1)))
        deep1 = self.deep_out(h1_small)

        h2 = self.dropout(F.relu(self.deep_fc1(x2)))
        h2 = self.dropout(F.relu(self.deep_fc2(h2)))
        h2_small = self.dropout(F.relu(self.deep_fc3(h2)))
        deep2 = self.deep_out(h2_small)

        logit1 = deep1 + self.wide(x1)
        logit2 = deep2 + self.wide(x2)

        if return_latent:
            return logit1, logit2, h1_small, h2_small

        return logit1, logit2

In [12]:
class TennisFusionModel(nn.Module):
    def __init__(self,
                 main_net: "SymmetricNnWideDeep",
                 form_input_size: int,
                 form_hidden: int = 64,
                 main_latent_dim: int = 64,
                 film_hidden: int = 128,
                 dropout: float = 0.3):
        super().__init__()
        self.main_net = main_net

        self.form_encoder = PairwiseFormEncoder(
            form_dim=form_input_size,
            hidden_size=form_hidden,
            dropout=0.0,
        )

        # FiLM from h_form -> gamma,beta for each dim of main latent
        self.film = nn.Sequential(
            nn.Linear(form_hidden, film_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(film_hidden, 2 * main_latent_dim),
        )

        self.form_head = nn.Sequential(
            nn.Linear(main_latent_dim + form_hidden + 1, film_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(film_hidden, 1),
        )

    def forward(self, p1_info_f, p1_form_seq, p2_info_f, p2_form_seq, env_f):
        logit1, logit2, h1, h2 = self.main_net(p1_info_f, p2_info_f, env_f, return_latent=True )
        base_logit = 0.5 * (logit1 - logit2)
        h_main = 0.5 * (h1 - h2)

        h_form = self.form_encoder(p1_form_seq, p2_form_seq)

        # FiLM parameters
        gamma_beta = self.film(h_form)               # (B,2D)
        gamma, beta = gamma_beta.chunk(2, dim=1)     # (B,D), (B,D)

        # modulate main latent
        h_mod = h_main * (1.0 + gamma) + beta        # (B,D)

        # correction logit (sees modulated main + form + base_logit)
        form_logit = self.form_head(torch.cat([h_mod, h_form, base_logit], dim=1))  # (B,1)

        final_logit = base_logit + form_logit
        prob = torch.sigmoid(final_logit)

        return prob

In [13]:
def build_model(params, player_info_feature_size, env_feature_size):
    main_net = SymmetricNnWideDeep(
        player_info_feature_size=player_info_feature_size,
        env_feature_size=env_feature_size,
        dropout=params["dropout"],
        bottleneck=params["bottleneck"],
        hidden_mid=params["hidden_mid"],
        hidden_small=params["hidden_small"],
    ).to(device)

    model = TennisFusionModel(
        main_net=main_net,
        form_input_size=params["form_input_size"],
        form_hidden=params["form_hidden_dim"],
        main_latent_dim=params["hidden_small"],
        film_hidden=params["fusion_hidden_dim"],
        dropout=params["fusion_dropout"]
    ).to(device)

    return model

In [14]:
from copy import deepcopy
from sklearn.metrics import brier_score_loss
from torch.optim.lr_scheduler import ReduceLROnPlateau
from models.data_loading import prepare_dataloaders, build_player_vocab
import joblib

player_vocab = build_player_vocab(matches)

def train_and_evaluate(params, train_matches, val_matches):

    player1_info_cols = params["player1_info_cols"]
    env_cols = params["env_cols"]

    batch_size = params["batch_size"]

    train_loader, val_loader, scalers = prepare_dataloaders(train_matches, val_matches, player1_info_cols, player2_info_cols, env_cols, batch_size, player_vocab, device)
    joblib.dump(scalers["env_scaler"], "env_scaler.pkl")
    joblib.dump(scalers["form_params"], "form_params.pkl")
    joblib.dump(scalers["info_residual_scaler"], "info_residual_scaler.pkl")
    joblib.dump(scalers["elo_scaler"], "elo_scaler.pkl")

    model = build_model(params, len(player1_info_cols), len(env_cols))

    def initialize_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    model.apply(initialize_weights)
    model.form_encoder.encoder.gru.flatten_parameters()

    opt_name = params["optimizer"]

    if opt_name == "Adam":
        optimizer = torch.optim.AdamW(model.parameters(), lr=params["learning_rate"],
                                      weight_decay=params["weight_decay"])
    elif opt_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=params["learning_rate"],
                                    weight_decay=params["weight_decay"], momentum=0.9)
    elif opt_name == "RMSprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=params["learning_rate"],
                                        weight_decay=params["weight_decay"])
    else:
        raise ValueError(f"Unknown optimizer {opt_name}")

    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.4)


    best_epoch_brier_score = float('inf')
    best_model_state = None

    criterion = nn.BCELoss()

    for epoch in range(params["epochs"]):
        model.train()
        for (p1_id, p2_id, p1_info, p1_form, p2_info, p2_form, env, labels) in train_loader:

            optimizer.zero_grad()
            pred  = model(p1_info, p1_form, p2_info, p2_form, env)

            loss = criterion(pred, labels.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        val_preds, val_truth = [], []
        with torch.no_grad():
            for (p1_id, p2_id,  p1_info, p1_form, p2_info, p2_form,env, labels) in val_loader:

                preds = model(p1_info, p1_form, p2_info, p2_form, env)
                labels = labels.float()
                #preds = torch.sigmoid(logit_preds)
                val_preds.append(preds.detach().cpu())
                val_truth.append(labels.cpu())


        val_preds = torch.cat(val_preds).numpy()
        val_truth = torch.cat(val_truth).numpy()

        brier = brier_score_loss(val_truth, val_preds)
        scheduler.step(brier)
        if brier < best_epoch_brier_score:
            best_epoch_brier_score = brier
            best_model_state = deepcopy(model.state_dict())

    best_model = deepcopy(model)

    best_model.load_state_dict(best_model_state)
    return best_epoch_brier_score, best_model

In [15]:
import numpy as np

def train_val_test(
    train_data,
    val_data,
    test_data,
    hyperparams,
    model=None
):

    best_val_score = None
    if not model:
        best_val_score, model = train_and_evaluate(
            hyperparams,
            train_data,
            val_data,
        )
        print(f"[train_val_test] Best Validation Score: {best_val_score:.4f}")

    player1_info_cols = hyperparams['player1_info_cols']
    player2_info_cols = hyperparams['player2_info_cols']
    env_cols = hyperparams['env_cols']

    _, _, test_loader, match_ids, scalers = prepare_dataloaders(
        train_data,
        val_data,
        p1_info_cols=player1_info_cols,
        p2_info_cols=player2_info_cols,
        env_cols=env_cols,
        batch_size=hyperparams['batch_size'],
        player_vocab=player_vocab,
        device=device,
        test_df=test_data
    )

    model.eval()
    with torch.no_grad():
        test_preds = []
        test_labels = []
        odds1_list = []
        odds2_list = []
        match_id_keys = []
        player1_ranks = []
        player2_ranks = []
        for (p1_id, p2_id, p1_info, p1_form, p2_info, p2_form, env,
            labels, p1_rank, p2_rank, odds1, odds2, match_id_key) in test_loader:
            preds  = model(p1_info, p1_form, p2_info, p2_form, env)
            match_id_key_strs = [str(match_ids[m]) for m in match_id_key]

            test_preds.extend(preds.squeeze(-1).tolist())

            labels = labels.float()
            test_labels.extend(labels.squeeze(-1).tolist())

            odds1_list.extend(odds1.tolist())
            odds2_list.extend(odds2.tolist())

            player1_ranks.extend(p1_rank.tolist())
            player2_ranks.extend(p2_rank.tolist())

            match_id_keys.extend(match_id_key.tolist())

    match_ids = [match_ids[i] for i in match_id_keys]
    odds1_list = np.array(odds1_list).flatten()
    odds2_list = np.array(odds2_list).flatten()
    test_preds = np.array(test_preds).flatten()
    test_labels = np.array(test_labels).flatten()
    match_ids = np.array(match_ids)
    player1_ranks = np.array(player1_ranks).flatten()
    player2_ranks = np.array(player2_ranks).flatten()

    valid_odds_mask = (~np.isnan(odds1_list)) & (~np.isnan(odds2_list))

    test_preds = test_preds[valid_odds_mask]
    test_labels = test_labels[valid_odds_mask]
    odds1_list  = odds1_list[valid_odds_mask]
    odds2_list  = odds2_list[valid_odds_mask]
    match_ids   = match_ids[valid_odds_mask]
    player1_ranks = player1_ranks[valid_odds_mask]
    player2_ranks = player2_ranks[valid_odds_mask]

    overall_model_brier = brier_score_loss(test_labels, test_preds)

    probW = 1.0 / odds1_list
    probL = 1.0 / odds2_list
    total_prob = probW + probL
    probW = np.divide(probW, total_prob, out=np.full_like(probW, 0.5), where=(total_prob != 0))
    probW = np.where(np.isnan(probW), 0.5, probW)

    overall_odds_brier = brier_score_loss(test_labels, probW)

    print(f"Overall Test Brier (Model): {overall_model_brier:.4f}")
    print(f"Overall Test Brier (Odds):  {overall_odds_brier:.4f}")

    results_df = pd.DataFrame({
        "match_id": match_ids.flatten(),
        "true_label": test_labels.flatten(),
        "model_prediction": test_preds.flatten(),
        "odds_prediction": probW.flatten(),
    })
    results_df["rank_less_50"] = (player1_ranks < 50) & (player2_ranks < 50)

    rank_mask = (player1_ranks < 50) & (player2_ranks < 50)
    filtered_preds   = test_preds[rank_mask]
    filtered_labels  = test_labels[rank_mask]
    filtered_odds1   = odds1_list[rank_mask]
    filtered_odds2   = odds2_list[rank_mask]

    filtered_model_brier = brier_score_loss(filtered_labels, filtered_preds)

    f_probW = 1.0 / filtered_odds1
    f_probL = 1.0 / filtered_odds2
    f_total = f_probW + f_probL
    f_probW = np.divide(f_probW, f_total, out=np.full_like(f_probW, 0.5), where=(f_total != 0))
    f_probW = np.where(np.isnan(f_probW), 0.5, f_probW)

    filtered_odds_brier = brier_score_loss(filtered_labels, f_probW)

    print(f"Filtered Test Brier (Model, player rank <= 50): {filtered_model_brier:.4f}")
    print(f"Filtered Test Brier (Odds,  player rank <= 50): {filtered_odds_brier:.4f}")

    return model,  {
        "best_val_score": best_val_score,
        "test_model_brier": filtered_model_brier,
        "test_odds_brier": filtered_odds_brier,
    }, results_df

In [16]:
config = {
    "player1_info_cols": player1_info_cols,
    "player2_info_cols": player2_info_cols,
    "form_input_size": 9,
    "env_cols": env_cols,
    'embedding_hidden_layers': [256, 128],
    'bottleneck': 512,
    'hidden_mid': 128,
    'hidden_small': 64,
    'form_hidden_dim': 32,
    'film_hidden': 128,
    'fusion_hidden_dim': 256,
    "dropout": 0.3417155781837491,
    "fusion_dropout": 0.4394839397565914,
    "learning_rate": 0.0008271984983392677,
    "weight_decay": 6.0649180089170696e-05,
    "batch_size": 256,
    "epochs": 50,
    "optimizer": "Adam"
}
best_model, results, match_predictions_df = train_val_test(
    train_data=train_matches,
    val_data=val_matches,
    test_data=test_matches,
    hyperparams=config
)

0.6637371035536874
[train_val_test] Best Validation Score: 0.2095
Overall Test Brier (Model): 0.2149
Overall Test Brier (Odds):  0.2037
Filtered Test Brier (Model, player rank <= 50): 0.1988
Filtered Test Brier (Odds,  player rank <= 50): 0.1917


F:\Pawel\studia_delft\Q2\ESL\MainProject2\.venv\lib\site-packages\torch\nn\modules\rnn.py:1392: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1410.)
  result = _VF.gru(


In [17]:
def year_based_cv(data, start_year=2015, end_year=2022, year_col="year", min_train_years=2):
    for val_year in range(start_year + min_train_years, end_year + 1):
        # Training years: from start_year to (val_year - 1)
        train_years = list(range(start_year, val_year))

        # Validation year: val_year
        train_idx = data[data[year_col].isin(train_years)].index
        val_idx = data[data[year_col] == val_year].index

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue
        yield train_idx, val_idx

In [18]:
def run_cv(train_data, hyperparams, start_year=2015, end_year=2022):
    scores = []
    for fold, (train_idx, val_idx) in enumerate(year_based_cv(train_data, start_year, end_year), 1):
        train_fold = train_data.loc[train_idx]
        val_fold   = train_data.loc[val_idx]
        fold_score, _ = train_and_evaluate(hyperparams, train_fold, val_fold)
        print(f"Fold {fold} best brier: {fold_score:.4f}")
        scores.append(fold_score)
    return np.mean(scores)

In [19]:
import optuna

def objective(trial):
    layer_options = {
        "small": [64, 32],
        "medium": [128, 64, 32],
        "large": [256, 128]
    }

    choice_key = trial.suggest_categorical("embedding_layers", list(layer_options.keys()))

    params = {
        "env_cols": env_cols,
        "player1_info_cols": player1_info_cols,
        "player2_info_cols": player2_info_cols,
        "form_input_size": 9,
        "epochs": 25,
        "seed": 42,

        # tunables
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "fusion_dropout": trial.suggest_float("fusion_dropout", 0.1, 0.5),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512]),
        "bottleneck": trial.suggest_categorical("bottleneck", [256, 512, 768]),
        "hidden_mid": trial.suggest_categorical("hidden_mid", [128, 256, 384]),
        "hidden_small": trial.suggest_categorical("hidden_small", [64, 128]),
        "form_hidden_dim": trial.suggest_categorical("form_hidden_dim", [32, 64, 128]),
        "film_hidden": trial.suggest_categorical("film_hidden", [64, 128, 256]),
        "fusion_hidden_dim": trial.suggest_categorical("fusion_hidden_dim", [64, 128, 256]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"]),
    }

    params["embedding_hidden_layers"] = layer_options[choice_key]

    brier = run_cv(train_matches, params, 2015, 2022)

    return brier


F:\Pawel\studia_delft\Q2\ESL\MainProject2\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
study = optuna.create_study(direction="minimize", study_name="tennis_tuning")
study.optimize(objective, n_trials=30)

print("Best trial:")
best = study.best_trial
print(f"  Brier score: {best.value:.4f}")
for k, v in best.params.items():
    print(f"  {k}: {v}")


[I 2026-01-25 20:58:58,787] A new study created in memory with name: tennis_tuning


0.666793893129771
Fold 1 best brier: 0.2110
0.6484641638225256
Fold 2 best brier: 0.2160
0.6508921644685803
Fold 3 best brier: 0.2175
0.6757188498402555
Fold 4 best brier: 0.2076


[I 2026-01-25 21:00:45,248] Trial 0 finished with value: 0.21355950500323545 and parameters: {'embedding_layers': 'medium', 'learning_rate': 8.934314228188586e-05, 'weight_decay': 1.2840451284373893e-05, 'dropout': 0.15094044520182792, 'fusion_dropout': 0.3521619199919047, 'batch_size': 128, 'bottleneck': 512, 'hidden_mid': 256, 'hidden_small': 64, 'form_hidden_dim': 128, 'film_hidden': 64, 'fusion_hidden_dim': 256, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.21355950500323545.


0.6531932093775262
Fold 5 best brier: 0.2157
0.6469465648854962
Fold 1 best brier: 0.2170
0.6412590064467197
Fold 2 best brier: 0.2224
0.6268425135764158
Fold 3 best brier: 0.2242
0.6333865814696485
Fold 4 best brier: 0.2244


[I 2026-01-25 21:02:29,901] Trial 1 finished with value: 0.2245272599774238 and parameters: {'embedding_layers': 'large', 'learning_rate': 4.267228659076106e-05, 'weight_decay': 2.0860309025809385e-06, 'dropout': 0.21501621105541693, 'fusion_dropout': 0.142229972950377, 'batch_size': 128, 'bottleneck': 512, 'hidden_mid': 128, 'hidden_small': 128, 'form_hidden_dim': 32, 'film_hidden': 256, 'fusion_hidden_dim': 256, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.21355950500323545.


...

0.6608730800323362
Fold 5 best brier: 0.2147
0.6717557251908397
Fold 1 best brier: 0.2093
0.6469472885855139
Fold 2 best brier: 0.2150
0.6532195500387897
Fold 3 best brier: 0.2171
0.6725239616613419
Fold 4 best brier: 0.2076


[I 2026-01-25 21:37:07,316] Trial 28 finished with value: 0.21295354072475167 and parameters: {'embedding_layers': 'large', 'learning_rate': 0.0003723208875328158, 'weight_decay': 0.00013927102251774844, 'dropout': 0.1830166057355802, 'fusion_dropout': 0.2151072430762883, 'batch_size': 128, 'bottleneck': 512, 'hidden_mid': 384, 'hidden_small': 64, 'form_hidden_dim': 32, 'film_hidden': 256, 'fusion_hidden_dim': 256, 'optimizer': 'Adam'}. Best is trial 25 with value: 0.2124267657842343.


0.64874696847211
Fold 5 best brier: 0.2157
0.6648854961832061
Fold 1 best brier: 0.2103
0.6522563519150549
Fold 2 best brier: 0.2144
0.652055857253685
Fold 3 best brier: 0.2172
0.6725239616613419
Fold 4 best brier: 0.2061


[I 2026-01-25 21:38:52,216] Trial 29 finished with value: 0.21245395057027525 and parameters: {'embedding_layers': 'medium', 'learning_rate': 0.0006831768614888148, 'weight_decay': 2.45076993488309e-05, 'dropout': 0.28966998297863045, 'fusion_dropout': 0.26568427426638774, 'batch_size': 128, 'bottleneck': 768, 'hidden_mid': 384, 'hidden_small': 64, 'form_hidden_dim': 64, 'film_hidden': 64, 'fusion_hidden_dim': 128, 'optimizer': 'RMSprop'}. Best is trial 25 with value: 0.2124267657842343.


0.6535974130962005
Fold 5 best brier: 0.2142
Best trial:
  Brier score: 0.2124
  embedding_layers: small
  learning_rate: 0.0005676517024518828
  weight_decay: 0.00015632480680827338
  dropout: 0.19153482644237635
  fusion_dropout: 0.19501027171861574
  batch_size: 128
  bottleneck: 768
  hidden_mid: 384
  hidden_small: 64
  form_hidden_dim: 128
  film_hidden: 256
  fusion_hidden_dim: 64
  optimizer: RMSprop
